# Paired image face-swap experiments

This first run uses the existing `data/DF40_face_crops/` dataset. The training code reads its saved face crops directly and does not prepare or download video data. Splits keep each fake/real pair, identity, and source video together. Both models use RGB inputs with the same transforms and training settings.

Command-line runs: `uv run python -m scripts.train.efficientnet_b0_face_swap` and `uv run python -m scripts.train.mobilenetv3_large_face_swap`.

In [ ]:
from look_again.paths import PROJECT_ROOT
from look_again.df40_subset import LEGACY_FACE_CROP_DATASET_PATH
from look_again.training import create_model, train_model

DATASET_ROOT = LEGACY_FACE_CROP_DATASET_PATH
print(f"Project: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_ROOT}")

## Select the existing face-crop dataset

This uses the face crops already saved under `data/DF40_face_crops/`. It will fail with a clear path error if the folder is missing or incomplete; it will not try to re-extract H.264 frames.

In [ ]:
face_dataset_path = LEGACY_FACE_CROP_DATASET_PATH
required_paths = (face_dataset_path / 'metadata.csv', face_dataset_path / 'fake', face_dataset_path / 'real')
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Existing face-crop dataset is incomplete: {missing_paths}')
print(f'Training data: {face_dataset_path}')

## Train both RGB baselines for 30 episodes

Both use the existing `data/DF40_face_crops/` folder, the same identity/video-disjoint split, Adam settings, and class-independent horizontal-flip/JPEG augmentation. Checkpoints are selected on validation data; the held-out test split is evaluated afterward.

In [ ]:
models = (
    ('efficientnet_b0', 'efficientnet_b0_existing_face_crops'),
    ('mobilenetv3_large_100', 'mobilenetv3_large_existing_face_crops'),
)
for model_name, run_name in models:
    model = create_model(model_name, face_swap_recipe=True)
    train_model(
        model,
        run_name,
        model_name=model_name,
        dataset_path=face_dataset_path,
        epochs=30,
        learning_rate=1e-4,
        weight_decay=0.01,
        train_augmentation=True,
        val_fraction=0.12,
        checkpoint_on='val',
        checkpoint_criterion='accuracy_minus_loss',
    )